In [ ]:
# input
pfam_pred_info = "../data/pfamId-numPro-numPredPro-numAnnoPro-isAnno.tsv"
pfam_info = "../../_database/Pfam/Pfam-A.clans.tsv"
# output
no_anno_pfam = "./data/no_anno_pfam.tsv"
pfam_fig = "./fig/pfam_categories.pdf"

In [2]:
import pandas as pd
df_pred = pd.read_table(pfam_pred_info, header=None, names=["pfam_id", "num_pro", "num_pred_pro", "num_anno_pro", "has_anno"])
df_info = pd.read_table(pfam_info, header=None, names=["pfam_id", "clan_id", "clan_name", "pfam_name", "pfam_desc"])
df_pred = df_pred[df_pred['num_anno_pro'] == 0]
df_pred = pd.merge(df_pred, df_info, on="pfam_id")
df_pred['pred_ratio'] = df_pred['num_pred_pro'] / df_pred["num_pro"]
# Pfam accession, clan accession, clan ID, PfamID, Pfam description
len(df_pred)

846

In [3]:
len(df_pred[df_pred['pfam_name'].map(lambda x: "DUF" in x)])

513

In [4]:
df_final = df_pred[df_pred.apply(lambda row: row['pred_ratio'] >= 0.5 and row['num_pro'] >= 10, axis=1)].copy()
len(df_final)

181

In [5]:
def has_metal_hint(x: str):
    x = x.lower()
    hints = {
        "zinc",
        "FeS4",
        "Metallo",
        "cobalt",
        "MerE",
        "FeGP"
    }

    for h in hints:
        h = h.lower()
        if x.count(h) > 0:
            return True
    return False

def is_unknown(row):
    if "DUF" in row['pfam_name'] or "UPF" in row['pfam_name']:
        return True
    if "unknown function" in row['pfam_desc']:
        return True
    return False

df_final['func_unknown'] = df_final.apply(lambda row: is_unknown(row), axis=1)
df_final['has_metal_hint'] = df_final['pfam_desc'].map(lambda x: has_metal_hint(x))
df_final.drop(columns=['num_anno_pro', 'has_anno'], inplace=True)
df_final.sort_values(by=["pfam_name"]).to_csv(no_anno_pfam, sep="\t", index=None)